# Two-Step Floating Catchment Area (2SFCA): Compute
## Pharmacy Access in Gauteng and KwaZulu-Natal

**Tess Vu**

Computes the 2SFCA pharmacy accessibility score for every SAL using
network-based routing. Derived from Jillian's `2sfca.ipynb` (archive/).
Maps and diagnostics live in `2sfca_visuals.ipynb`.

Working CRS: EPSG:32735 (UTM 35S, meters). Distances in meters.
Constants live in `config/analysis.yml`.

**Score Meaning:** higher = better access; 0 = no registered pharmacy reachable within the catchment on the road network.

**Two Steps:**
1. Supply: per pharmacy, decay-weighted population within catchment → ratio Rj.
2. Demand: per SAL, sum reachable pharmacies' Rj, decay-weighted → score Ai.

Four runs with asymmetric catchments. KZN is ~7x larger and far less dense, and rural residents routinely travel farther for medicine:
Gauteng 2 km walk / 5 km drive; KZN 3 km walk / 10 km drive.

- Output: `data/tess_all_access.csv` (38,380 rows; adds Ai_walk, Ai_drive, node)
- Runtime: graph loading + four Dijkstra sweeps — hours. Do not re-run casually.

In [3]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())))

import geopandas as gpd
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
from tqdm import tqdm

from src.config import load_config
from src.crs import CRS_UTM35S, CRS_WGS84
from src.paths import NETWORKS, PHARMACIES_MASTER, POP_PRED_FINAL, SAL_W_WARD_DEDUP, TESS_ALL_ACCESS

CFG = load_config()
BETA = CFG["decay"]["beta"]
MIN_WEIGHTED_POP = CFG["2sfca"]["min_weighted_pop"]
CUTOFFS_M = {
    ("Gauteng", "walk"):  CFG["catchments_km"]["gauteng"]["walk"] * 1000,
    ("Gauteng", "drive"): CFG["catchments_km"]["gauteng"]["drive"] * 1000,
    ("KwaZulu-Natal", "walk"):  CFG["catchments_km"]["kwazulu_natal"]["walk"] * 1000,
    ("KwaZulu-Natal", "drive"): CFG["catchments_km"]["kwazulu_natal"]["drive"] * 1000,
}
print(f"beta={BETA}, min_weighted_pop={MIN_WEIGHTED_POP}")
print(f"cutoffs (m): {CUTOFFS_M}")

beta=0.0003, min_weighted_pop=50
cutoffs (m): {('Gauteng', 'walk'): 2000, ('Gauteng', 'drive'): 5000, ('KwaZulu-Natal', 'walk'): 3000, ('KwaZulu-Natal', 'drive'): 10000}


## Load Source Data

- `pop_pred_final.csv`: ~38k SALs with 2023 dasymetric population estimates
- `PHARMACIES_MASTER_FINAL.csv`: deduplicated geocoded pharmacies (LAT/LNG, EPSG:4326)
- `sal_w_ward_dedup.shp`: SAL boundaries (EPSG:32735)

In [ ]:
sal_raw = pd.read_csv(POP_PRED_FINAL)
pharmacy_raw = pd.read_csv(PHARMACIES_MASTER)
sal_polygons = gpd.read_file(SAL_W_WARD_DEDUP)

pharmacy_gdf = gpd.GeoDataFrame(
    pharmacy_raw,
    geometry=gpd.points_from_xy(pharmacy_raw["LNG"], pharmacy_raw["LAT"]),
    crs=CRS_WGS84,
)
pharmacy_gdf = pharmacy_gdf.to_crs(sal_polygons.crs)

# Normalize EA_CODE to int64 on both sides to avoid int/float merge warning.
sal_raw["EA_CODE"] = pd.to_numeric(sal_raw["EA_CODE"], errors="coerce").astype("int64")
sal_polygons["EA_CODE"] = pd.to_numeric(sal_polygons["EA_CODE"], errors="coerce").astype("int64")

sal = sal_raw.merge(
    sal_polygons[["EA_CODE", "PR_NAME", "geometry"]],
    on="EA_CODE",
    how="left",
)
sal = gpd.GeoDataFrame(sal, geometry="geometry")
print(f"SALs: {len(sal)}  |  Pharmacies: {len(pharmacy_gdf)}")

## Variable Catchment (optional, not enabled)

Per-settlement-type catchments (Urban 2 km, Traditional 10 km, Other 15 km) are an alternative to fixed per-province cutoffs. Needs a closer look at the settlement-type distribution first. If enabled, the 2SFCA calls would read per-SAL catchments instead of the fixed `CUTOFFS_M` values.

In [ ]:
# def assign_catchment(row):
#     if row["EA_GTYPE"] == "Urban":
#         return 2000
#     elif row["EA_GTYPE"] == "Traditional":
#         return 10000
#     return 15000
# sal["catchment"] = sal.apply(assign_catchment, axis=1)

In [ ]:
# Split by province; spatial join assigns pharmacies to their province.
kzn_sal = sal[sal["PR_NAME"] == "KwaZulu-Natal"].copy()
gau_sal = sal[sal["PR_NAME"] == "Gauteng"].copy()

kzn_pharmacies = gpd.sjoin(pharmacy_gdf, kzn_sal, predicate="within")
gau_pharmacies = gpd.sjoin(pharmacy_gdf, gau_sal, predicate="within")

print(f"KZN SALs: {len(kzn_sal)}  |  KZN Pharmacies: {len(kzn_pharmacies)}")
print(f"GAU SALs: {len(gau_sal)}  |  GAU Pharmacies: {len(gau_pharmacies)}")

## Distance Decay and 2SFCA

beta = 0.0003 (was 0.0001, which barely distinguished near from far, so a pharmacy 10 km out still got 37% weight). At 0.0003: 1 km → 0.741,
2 km → 0.549, 5 km → 0.223, 10 km → 0.050.

`run_2sfca` carries two bug fixes from the previous version:
- **Node Collision:** `.groupby("node").sum()` for the population lookup,  the old `.set_index("node")` silently kept only one SAL's population when several SALs snapped to the same road node.
- **Rj Outlier Cap:** a floor of `min_weighted_pop` on the catchment population prevents near-infinite ratios (a 3-person catchment produced Rj = 333, ~5,000x the mean).

In [ ]:
def decay(distance, beta=BETA):
    return np.exp(-beta * distance)


def run_2sfca(sal_gdf, pharm_gdf, graph, cutoff_meters, score_column):
    """Both 2SFCA steps for one province/mode. Requires 'node' and 'sal2023_est'."""
    pop_dict = sal_gdf.groupby("node")["sal2023_est"].sum().to_dict()

    print(f"STEP 1: Computing Rj for {len(pharm_gdf)} pharmacies (cutoff = {cutoff_meters}m)")
    rj_list = []
    for idx, pharm in tqdm(pharm_gdf.iterrows(), total=len(pharm_gdf)):
        lengths = nx.single_source_dijkstra_path_length(
            graph, pharm["node"], cutoff=cutoff_meters, weight="length"
        )
        weighted_pop = sum(
            pop_dict.get(node, 0) * decay(dist)
            for node, dist in lengths.items()
        )
        if weighted_pop > 0:
            rj = 1 / (max(weighted_pop, MIN_WEIGHTED_POP) / 1000)
        else:
            rj = 0
        rj_list.append(rj)

    pharm_gdf["Rj"] = rj_list
    pharm_dict = pharm_gdf.groupby("node")["Rj"].sum().to_dict()

    print(f"STEP 2: Computing Ai for {len(sal_gdf)} SALs")
    ai_list = []
    for idx, pop in tqdm(sal_gdf.iterrows(), total=len(sal_gdf)):
        lengths = nx.single_source_dijkstra_path_length(
            graph, pop["node"], cutoff=cutoff_meters, weight="length"
        )
        ai = sum(
            pharm_dict.get(node, 0) * decay(dist)
            for node, dist in lengths.items()
        )
        ai_list.append(ai)

    sal_gdf[score_column] = ai_list
    print(f"COMPLETE: {score_column}")
    print(sal_gdf[score_column].describe())
    return sal_gdf

## KwaZulu-Natal: Walk

Graph loading is the slow step, load each raw graph once and do not re-run `load_graphml` cells unnecessarily.

In [ ]:
kzn_walk_graph_raw = ox.load_graphml(NETWORKS / "network_kwazulu_natal_walk.graphml")
print(f"KZN walk RAW: {kzn_walk_graph_raw.number_of_nodes()} nodes, {kzn_walk_graph_raw.number_of_edges()} edges")

In [ ]:
kzn_walk_graph = ox.project_graph(kzn_walk_graph_raw, to_crs=CRS_UTM35S)
print(f"KZN walk PROJECTED: {kzn_walk_graph.number_of_nodes()} nodes, {kzn_walk_graph.number_of_edges()} edges")

kzn_sal = kzn_sal.to_crs(kzn_walk_graph.graph["crs"])
kzn_sal = kzn_sal.set_geometry(kzn_sal.geometry.centroid)
kzn_pharmacies = kzn_pharmacies.to_crs(kzn_walk_graph.graph["crs"])

kzn_sal["node"] = ox.nearest_nodes(kzn_walk_graph, kzn_sal.geometry.x, kzn_sal.geometry.y)
kzn_walk_pharm = kzn_pharmacies.copy()
kzn_walk_pharm["node"] = ox.nearest_nodes(kzn_walk_graph, kzn_walk_pharm.geometry.x, kzn_walk_pharm.geometry.y)

In [ ]:
kzn_sal = run_2sfca(kzn_sal, kzn_walk_pharm, kzn_walk_graph,
                    cutoff_meters=CUTOFFS_M[("KwaZulu-Natal", "walk")], score_column="Ai_walk")
kzn_sal["Ai_walk"].describe(percentiles=[0.9, 0.95, 0.98, 0.99, 0.999])

## KwaZulu-Natal: Drive

In [ ]:
kzn_drive_graph_raw = ox.load_graphml(NETWORKS / "network_kwazulu_natal_drive.graphml")
print(f"KZN drive RAW: {kzn_drive_graph_raw.number_of_nodes()} nodes, {kzn_drive_graph_raw.number_of_edges()} edges")

In [ ]:
kzn_drive_graph = ox.project_graph(kzn_drive_graph_raw, to_crs=CRS_UTM35S)
print(f"KZN drive PROJECTED: {kzn_drive_graph.number_of_nodes()} nodes, {kzn_drive_graph.number_of_edges()} edges")

kzn_sal = kzn_sal.to_crs(kzn_drive_graph.graph["crs"])
kzn_pharmacies = kzn_pharmacies.to_crs(kzn_drive_graph.graph["crs"])

kzn_sal["node"] = ox.nearest_nodes(kzn_drive_graph, kzn_sal.geometry.x, kzn_sal.geometry.y)
kzn_drive_pharm = kzn_pharmacies.copy()
kzn_drive_pharm["node"] = ox.nearest_nodes(kzn_drive_graph, kzn_drive_pharm.geometry.x, kzn_drive_pharm.geometry.y)

In [ ]:
kzn_sal = run_2sfca(kzn_sal, kzn_drive_pharm, kzn_drive_graph,
                    cutoff_meters=CUTOFFS_M[("KwaZulu-Natal", "drive")], score_column="Ai_drive")
kzn_sal["Ai_drive"].describe(percentiles=[0.9, 0.95, 0.98, 0.99, 0.999])

## Gauteng: Walk

In [ ]:
gau_walk_graph_raw = ox.load_graphml(NETWORKS / "network_gauteng_walk.graphml")
print(f"GAU walk RAW: {gau_walk_graph_raw.number_of_nodes()} nodes, {gau_walk_graph_raw.number_of_edges()} edges")

In [ ]:
gau_walk_graph = ox.project_graph(gau_walk_graph_raw, to_crs=CRS_UTM35S)
print(f"GAU walk PROJECTED: {gau_walk_graph.number_of_nodes()} nodes, {gau_walk_graph.number_of_edges()} edges")

gau_sal = gau_sal.to_crs(gau_walk_graph.graph["crs"])
gau_sal = gau_sal.set_geometry(gau_sal.geometry.centroid)
gau_pharmacies = gau_pharmacies.to_crs(gau_walk_graph.graph["crs"])

gau_sal["node"] = ox.nearest_nodes(gau_walk_graph, gau_sal.geometry.x, gau_sal.geometry.y)
gau_walk_pharm = gau_pharmacies.copy()
gau_walk_pharm["node"] = ox.nearest_nodes(gau_walk_graph, gau_walk_pharm.geometry.x, gau_walk_pharm.geometry.y)

In [ ]:
gau_sal = run_2sfca(gau_sal, gau_walk_pharm, gau_walk_graph,
                    cutoff_meters=CUTOFFS_M[("Gauteng", "walk")], score_column="Ai_walk")
gau_sal["Ai_walk"].describe(percentiles=[0.9, 0.95, 0.98, 0.99, 0.999])

## Gauteng: Drive

In [ ]:
gau_drive_graph_raw = ox.load_graphml(NETWORKS / "network_gauteng_drive.graphml")
print(f"GAU drive RAW: {gau_drive_graph_raw.number_of_nodes()} nodes, {gau_drive_graph_raw.number_of_edges()} edges")

In [ ]:
# Explicit CRS: the original relied on auto-UTM, which resolves to 32735 here.
gau_drive_graph = ox.project_graph(gau_drive_graph_raw, to_crs=CRS_UTM35S)
print(f"GAU drive PROJECTED: {gau_drive_graph.number_of_nodes()} nodes, {gau_drive_graph.number_of_edges()} edges")

gau_sal = gau_sal.to_crs(gau_drive_graph.graph["crs"])
gau_pharmacies = gau_pharmacies.to_crs(gau_drive_graph.graph["crs"])

gau_sal["node"] = ox.nearest_nodes(gau_drive_graph, gau_sal.geometry.x, gau_sal.geometry.y)
gau_drive_pharm = gau_pharmacies.copy()
gau_drive_pharm["node"] = ox.nearest_nodes(gau_drive_graph, gau_drive_pharm.geometry.x, gau_drive_pharm.geometry.y)

In [ ]:
gau_sal = run_2sfca(gau_sal, gau_drive_pharm, gau_drive_graph,
                    cutoff_meters=CUTOFFS_M[("Gauteng", "drive")], score_column="Ai_drive")
gau_sal["Ai_drive"].describe(percentiles=[0.9, 0.95, 0.98, 0.99, 0.999])

## Combine Provinces and Export

In [ ]:
print("DUPLICATE EA_CODES IN SAL_POLYGONS:")
dup_mask = sal_polygons.duplicated(subset="EA_CODE", keep=False)
print(f"Total duplicate rows: {dup_mask.sum()}")
print(f"Unique EA_CODEs with duplicates: {sal_polygons.loc[dup_mask, 'EA_CODE'].nunique()}")

In [ ]:
# Percentile ranks within each province; log transforms for mapping.
for province_sal in [kzn_sal, gau_sal]:
    for col in ["Ai_walk", "Ai_drive"]:
        rank_col = col.replace("Ai_", "") + "_rank"
        province_sal[rank_col] = 0.0
        nonzero = province_sal[col] > 0
        province_sal.loc[nonzero, rank_col] = province_sal.loc[nonzero, col].rank(pct=True)
    province_sal["walk_log"] = np.log1p(province_sal["Ai_walk"])
    province_sal["drive_log"] = np.log1p(province_sal["Ai_drive"])

allaccess = pd.concat([kzn_sal, gau_sal], ignore_index=True)
allaccess = allaccess.drop(columns="geometry")

# keep="last" retains rows with both Ai_walk and Ai_drive populated.
allaccess = allaccess.drop_duplicates(subset="EA_CODE", keep="last")

allaccess = sal_polygons.merge(allaccess, on="EA_CODE", how="left")
print(f"COMBINED DATASET AFTER DEDUPLICATION: {len(allaccess)} SALs")

score_cols = ["walk_log", "drive_log", "walk_rank", "drive_rank"]
allaccess[score_cols] = allaccess[score_cols].fillna(0)
for c in score_cols:
    allaccess[c] = pd.to_numeric(allaccess[c], errors="coerce")

# Resolve duplicate-suffix columns from the merges.
x_cols = [c for c in allaccess.columns if c.endswith("_x")]
y_cols = [c for c in allaccess.columns if c.endswith("_y")]
allaccess = allaccess.drop(columns=y_cols)
allaccess = allaccess.rename(columns={c: c[:-2] for c in x_cols})

In [ ]:
allaccess_export = allaccess.drop(columns="geometry")
# Mixed-type artifacts from the SAL census data; cast to keep them.
allaccess_export["smallplace"] = allaccess_export["smallplace"].astype(str)
allaccess_export.to_csv(
    TESS_ALL_ACCESS,
    index=False,
    float_format="%.6f",
    na_rep="",
)
print(f"EXPORTED: {TESS_ALL_ACCESS}")

## Categorical Accessibility Classification (diagnostic, not in CSV)

Tiers are computed within each province because the catchment thresholds are intentionally asymmetric. Ai = 0 folds into Pharmacy Desert.

| Tier | Definition |
|------|-----------|
| Pharmacy Desert | Ai = 0 or bottom third of non-zero scores |
| Moderate Access | Middle third of non-zero scores |
| Served | Top third of non-zero scores |

In [ ]:
def classify_access(df, score_col, tier_col):
    df[tier_col] = "Pharmacy Desert"
    nonzero_mask = df[score_col] > 0
    nonzero_scores = df.loc[nonzero_mask, score_col]
    if len(nonzero_scores) == 0:
        return df

    t33 = nonzero_scores.quantile(0.333)
    t66 = nonzero_scores.quantile(0.666)
    df.loc[nonzero_mask & (df[score_col] <= t33), tier_col] = "Pharmacy Desert"
    df.loc[nonzero_mask & (df[score_col] > t33) & (df[score_col] <= t66), tier_col] = "Moderate Access"
    df.loc[nonzero_mask & (df[score_col] > t66), tier_col] = "Served"
    return df


allaccess["walk_tier"] = "Pharmacy Desert"
allaccess["drive_tier"] = "Pharmacy Desert"

for prov in allaccess["PR_NAME"].dropna().unique():
    prov_mask = allaccess["PR_NAME"] == prov
    classified = classify_access(allaccess.loc[prov_mask].copy(), "Ai_walk", "walk_tier")
    allaccess.loc[prov_mask, "walk_tier"] = classified["walk_tier"].values
    classified = classify_access(allaccess.loc[prov_mask].copy(), "Ai_drive", "drive_tier")
    allaccess.loc[prov_mask, "drive_tier"] = classified["drive_tier"].values

tier_order = ["Pharmacy Desert", "Moderate Access", "Served"]
allaccess["walk_tier"] = pd.Categorical(allaccess["walk_tier"], categories=tier_order, ordered=True)
allaccess["drive_tier"] = pd.Categorical(allaccess["drive_tier"], categories=tier_order, ordered=True)

print("WALK TIER COUNTS BY PROVINCE")
print(allaccess.groupby("PR_NAME")["walk_tier"].value_counts().unstack(fill_value=0))
print()
print("DRIVE TIER COUNTS BY PROVINCE")
print(allaccess.groupby("PR_NAME")["drive_tier"].value_counts().unstack(fill_value=0))
print()
print("WALK TIER PERCENTAGES BY PROVINCE")
print((allaccess.groupby("PR_NAME")["walk_tier"].value_counts(normalize=True).unstack(fill_value=0) * 100).round(1))
print()
print("DRIVE TIER PERCENTAGES BY PROVINCE")
print((allaccess.groupby("PR_NAME")["drive_tier"].value_counts(normalize=True).unstack(fill_value=0) * 100).round(1))

## NOTES AND LIMITATIONS

#### **Why two metrics are necessary (this notebook and `network_threshold.ipynb`).**

The 2SFCA $A_{i}$ score and the network k = 1 distance threshold answer different questions and have opposing failures, and *neither are sufficient alone*.

| Dimension | 2SFCA $A_{i}$ | Network Distance (k = 1) |
|---|---|---|
| What it measures. | Supply-to-demand ratio within catchment, decay-weighted. | Absolute distance on road network to nearest pharmacy. |
| Output scale. | Relative, so **only meaningful within province and travel mode**. | Absolute in meters/km, **cross-province comparable**. |
| What zeros mean. | No pharmacy reachable within catchment radius. | Does not produce zeros, always returns a finite distance. |
| High score pathology. | **Inflated by low-demand areas** (industrial, farms, sparse rural), could consider using different thresholds for EA_TYPEs, but adds complexity. | None, short distance is always genuinely short. |
| Demand pressure captured. | Yes, competing population in Step 1 denominator. | No. |
| Supply concentration visible. | Yes, one overwhelmed pharmacy scores lower than three dispersed ones. | No, only distance to nearest. |
| Zero-inflation problem. | Severe as **more than half of KZN walk SALs score exactly 0**. | Not applicable. |
| Modal dependency. | Walk and drive modeled separately. | Walk, drive, and Euclidean modeled separately. |
| Apartheid spatial signal. | Indirect via inherited pharmacy distribution. | More direct, rural SALs show largest absolute distances. |

**The k=3 distance metric addresses the floor problem** by providing an absolute, non-relative measure of how far a SAL sits from any pharmacy at all. When the $A_{i}$ is zero, the distance score tells you whether the SAL is zero because it is genuinely isolated (large distance) or because the road-network connectivity broke down near the catchment boundary (short distance but still zero $A_{i}$, which is a data quality flag, not a real desert). When the $A_{i}$ is artificially high (low-demand near-supply artifacts), the distance score anchors interpretation: **a SAL with a high $A_{i}$ but a long absolute distance is not genuinely accessible**.

---

#### **Access typology on combining $A_{i}$ w/ distance threshold.**

For the integration notebook `combine_access_score_network_threshold.ipynb`, the combination of these two metrics produces four interpretable quadrant types. The thresholds used below are the policy-relevant reference points from the network threshold notebook (3 km walk, 10 km drive).

| | Distance < threshold (pharmacy nearby) | Distance ≥ threshold (pharmacy far) |
|---|---|---|
| **$A_{i} = 0$** | Connectivity gap as pharmacy exists nearby but is road-network unreachable from this SAL. Likely an OSM completeness issue or a catchment boundary edge case. Warrants spot-check. | **True pharmacy desert** so no pharmacy reachable on the network and nearest pharmacy is also far in absolute terms. Primary NHI infrastructure target. |
| **$A_{i}$ low (bottom tercile, nonzero)** | Demand overcrowding where pharmacy is nearby but the supply-to-demand ratio is poor. Pharmacy exists but is serving a large population. Candidate for NHI capacity investment at the existing site. | Access gap, so far from a pharmacy and that pharmacy is also under-supplied relative to demand. Secondary infrastructure target. |
| **$A_{i}$ high (top tercile)** | Genuinely well-served w/ pharmacy nearby and the supply-to-demand ratio is favorable. | Artifact zone, so high $A_{i}$ is mechanically driven by low local demand (industrial, sparsely populated rural). Resident is actually far from a pharmacy. Score is not policy-relevant, so use distance as primary indicator. |

The artifact zone (high $A_{i}$, far distance) accounts for the pathological top-scoring SALs in both provinces: Isando and Zwavelpoort in Gauteng walk, and the Malunga/Smukumuku cluster in KZN drive. These should be flagged and excluded from the "well-served" narrative.

---

#### **Side-by-side numbers to know what each metric says about the same population.**

All figures from executed notebook outputs. Percentages are SAL-count-based unless marked (pop).

| Province | Mode | $A_{i}$: % Pharmacy Desert | Distance: % exceeding policy threshold | Policy threshold used |
|---|---|---|---|---|
| Gauteng | Walk | 53.2% | 24.6% (25.7% pop-weighted) | 3 km walk |
| KwaZulu-Natal | Walk | 74.3% | 63.1% (63.2% pop-weighted) | 3 km walk |
| Gauteng | Drive | 38.7% | 2.1% (1.1% pop-weighted) | 10 km drive |
| KwaZulu-Natal | Drive | 58.8% | 38.5% (35.4% pop-weighted) | 10 km drive |

The gap between the $A_{i}$ desert rate and the distance exceedance rate is the most informative number in this table bc for Gauteng walk, 53.2% of SALs are in the Pharmacy Desert tier but only 24.6% exceed 3 km. That 28-point gap is almost entirely zero-inflation where SALs that technically have *some* pharmacy within 2 km of their centroid but whose $A_{i}$ is zero because the nearest-node snap may have placed them outside any pharmacy's catchment. These are not true deserts in the same sense as Evaton North (dense, zero $A_{i}$, short distance but overwhelmed nearby pharmacy). The $A_{i}$ tier classification obscures this.

For KZN drive, the gap narrows to 20 points (58.8% desert tier, 38.5% exceeding 10 km). This is a more honest match as KZN drive deserts may be more likely to be genuinely isolated and not snap artifacts. The median drive distance for Traditional settlement KZN is 18.7 km, which is well beyond the 10 km policy threshold.

**Settlement type breakdown at policy thresholds** (from `network_threshold.ipynb`):

| Settlement | Province | % exceeding 3 km walk | % exceeding 10 km drive |
|---|---|---|---|
| Urban | Gauteng | ~17% | ~2% |
| Urban | KZN | ~25–30% | ~15% |
| Traditional | Gauteng | ~40–50% | ~5% |
| Traditional | KZN | ~75–80% | ~50% |
| Farms | Gauteng | ~60% | ~10% |
| Farms | KZN | ~80%+ | ~50%+ |

**Economic status breakdown at policy thresholds** (from `network_threshold.ipynb`, SAL-level):

| Status | Province | % exceeding 3 km walk | % exceeding 10 km drive |
|---|---|---|---|
| Wealthy | Gauteng | 17.6% | 2.8% |
| Non_Wealthy | Gauteng | 29.8% | 1.3% |
| Wealthy | KZN | 39.6% | 15.4% |
| Non_Wealthy | KZN | 74.9% | 49.6% |

The Non_Wealthy/Traditional KZN combination is the equity signal that directly connects to apartheid-era spatial planning. These households face the largest absolute distances and, per the $A_{i}$ scores, have essentially no pharmacy access on either walking or driving networks within the policy thresholds. These are also the areas least likely to have private vehicle access, meaning the drive score is aspirational rather than descriptive for them.